# 00 — Sweep

Lancia le combinazioni della matrice con Hydra, uno stadio alla volta:
train → export → benchmark → aggregazione. Il notebook non misura niente da
solo: compone gli override e chiama `run.py -m` in un sottoprocesso, lo
stesso comando che scriveresti a mano. Così ogni job gira in un processo
pulito e l'output di Hydra (log, `multirun/`) resta quello di sempre.

Due regole:

- **`DRY_RUN = True` di default.** Con "Run All" vedi quali celle verrebbero
  fatte, senza lanciare ore di lavoro. Mettilo a `False` solo quando la
  matrice è quella giusta.
- **Mai `hydra/launcher=joblib` nel benchmark.** Job concorrenti sulla stessa
  GPU o sugli stessi core producono latenze inservibili.

## 1. Setup

Trova la root del repo e prepara gli import.

In [ ]:
import shlex
import subprocess
import sys
from pathlib import Path

from omegaconf import OmegaConf

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CONF = ROOT / "conf"
assert (ROOT / "run.py").exists(), f"run.py non trovato in {ROOT}"
ROOT

## 2. Helper per lanciare Hydra

`hydra()` trasforma gli assi in override (le liste diventano `a,b,c`), lancia
`run.py` e stampa l'output riga per riga mentre arriva. Se il processo esce
con errore solleva, così la cella si ferma in rosso invece di sembrare riuscita.

In [ ]:
# Flag del tool: hanno un default in conf/config.yaml e si passano a hydra()
# come argomenti, es. hydra(..., retry_failed=True, allow_reboot=True).
# Riferimento completo: docs/config/config.yaml.
FLAGS = {
    "dry_run",            # simula: stampa cosa farebbe, senza eseguire (default false)
    "force",              # benchmark: rifà anche le celle già completate (default false)
    "retry_failed",       # benchmark: riprova le celle fallite, salta ok e skipped (default false)
    "force_retrain",      # training: riallena anche se i pesi sono già in cache (default false)
    "force_reexport",     # export: riesporta anche se l'artefatto esiste già (default false)
    "allow_reboot",       # Jetson: riavvia la board se il cambio di profilo lo richiede (default false)
    "skip_invalid",       # combinazione impossibile -> salvata come skipped e si prosegue; false = stop (default true)
    "continue_on_error",  # cella fallita -> salvata come failed e si prosegue; false = stop (default true)
}


def overrides(**axes) -> list[str]:
    """{"model": ["yolo26n", "yolo26s"]} -> ["model=yolo26n,yolo26s"].

    None viene ignorato; i flag booleani diventano `++flag=true`.
    """
    out = []
    for key, value in axes.items():
        if value is None:
            continue
        if isinstance(value, bool):
            value = str(value).lower()
        elif isinstance(value, (list, tuple)):
            value = ",".join(map(str, value))
        prefix = "++" if key in FLAGS else ""
        out.append(f"{prefix}{key}={value}")
    return out


def hydra(*extra: str, multirun: bool = True, dry_run: bool | None = None,
          **axes) -> None:
    """Lancia `python run.py [-m] <override>` dalla root del repo."""
    if dry_run is None:
        dry_run = DRY_RUN
    if dry_run and "--cfg" not in extra:
        axes["dry_run"] = True
    cmd = [sys.executable, "run.py"]
    if multirun:
        cmd.append("-m")
    cmd += overrides(**axes) + list(extra)
    print("$", shlex.join(cmd[1:]), "\n", flush=True)

    proc = subprocess.Popen(cmd, cwd=ROOT, text=True, bufsize=1,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"run.py uscito con codice {proc.returncode}")

## 3. Cosa c'è a disposizione

Elenca i file di ogni gruppo di `conf/` e, per ogni board, i profili di
frequenza e i compute target che dichiara: sono gli unici valori validi per
`freq_target` e `compute_target`.

In [ ]:
for group in ["model", "quantization", "backend", "hardware", "train", "stage"]:
    names = sorted(p.stem for p in (CONF / group).glob("*.yaml"))
    print(f"{group:13s} {', '.join(names)}")

print()
for path in sorted((CONF / "hardware").glob("*.yaml")):
    hw = OmegaConf.load(path)
    print(f"{path.stem:13s} freq={list(hw.freq)}  compute={list(hw.compute)}")

## 4. Parametri della matrice

L'unica cella da modificare: scegli i valori di ogni asse. Le liste
diventano sweep Hydra; `None` su `compute_target` vuol dire "tutti quelli
che la board dichiara".

In [ ]:
DRY_RUN = False          # False per eseguire davvero

MODELS = ["yolo26n"] #["yolo26n", "yolo26s"]
QUANTS = ["fp32", "fp16", "int8"]

# Workstation
WKS_BACKENDS = ["onnxruntime"] #["onnxruntime", "tensorrt", "openvino"]
WKS_COMPUTE = None                      # gpu, cpu_1, cpu_all

# Jetson: un sweep per profilo, perché certi cambi vogliono il riavvio
JETSON_BACKENDS = ["onnxruntime", "tensorrt", "openvino"]
JETSON_FREQS = ["maxn", "w30", "w15"]
JETSON_COMPUTE = None                   # gpu, cpu_1..cpu_8

# Raspberry Pi 5
RPI_BACKENDS = ["onnxruntime", "openvino", "axelera"]
RPI_FREQS = ["max", "mid", "low"]
RPI_COMPUTE = None                      # cpu_1, cpu_2, cpu_4, axelera

## 5. Anteprima della config risolta

Stampa la config di una singola combinazione (il primo valore di ogni asse)
con `--cfg job --resolve`, senza eseguire niente: serve a controllare che gli
interpolatori e i default siano quelli attesi.

In [ ]:
hydra("--cfg", "job", "--resolve", multirun=False,
      stage="train", model=MODELS[0], quantization=QUANTS[0],
      backend=WKS_BACKENDS[0], hardware="wks4_rtx6000")

## 6. Dry-run del benchmark

Stampa una riga per cella con l'esito previsto (`ok` o `skip` con il motivo)
per tutte e tre le board, qualunque sia il valore di `DRY_RUN`. È il momento
di accorgersi di una matrice sbagliata, prima di lanciarla.

In [ ]:
common = dict(stage="benchmark", model=MODELS, quantization=QUANTS)
hydra(dry_run=True, **common, backend=WKS_BACKENDS,
      hardware="wks4_rtx6000", compute_target=WKS_COMPUTE)
hydra(dry_run=True, **common, backend=JETSON_BACKENDS,
      hardware="jetson_orin", freq_target=JETSON_FREQS,
      compute_target=JETSON_COMPUTE)
hydra(dry_run=True, **common, backend=RPI_BACKENDS,
      hardware="rpi5", freq_target=RPI_FREQS, compute_target=RPI_COMPUTE)

## 7. Training

Allena ogni modello una volta sola sulla workstation; i modelli già in cache
vengono saltati. Qui il parallelismo con joblib sarebbe lecito, ma su una
sola GPU non conviene.

In [ ]:
hydra(stage="train", model=MODELS, dry_run = False, multirun=True)

## 8. Export portabile (workstation)

Produce gli ONNX e gli IR OpenVINO sulla workstation, con la validazione
numerica contro l'FP32. Due export distinti: uno per la workstation (x86_64)
e uno per le board (aarch64, vale sia per il Jetson sia per il Pi 5), perché
l'architettura fa parte dell'`export_key`.

In [ ]:
NOT_PORTABLE = {"tensorrt", "axelera"}   # compilati sulla board, stadio a parte

# L'export_key contiene l'architettura: gli artefatti per le board aarch64 sono
# export distinti da quelli x86, anche se prodotti qui sulla workstation. Jetson
# e Pi 5 sono entrambi aarch64, quindi basta un export per tutti e due.
x86 = sorted(set(WKS_BACKENDS) - NOT_PORTABLE)
arm = sorted(set(JETSON_BACKENDS + RPI_BACKENDS) - NOT_PORTABLE)
if x86:
    hydra(stage="export", model=MODELS, quantization=QUANTS, backend=x86,
          hardware="wks4_rtx6000", dry_run=False)
if arm:
    hydra(stage="export", model=MODELS, quantization=QUANTS, backend=arm,
          hardware="jetson_orin", dry_run=False)

## 9. Provisioning delle board

Prepara l'ambiente su Jetson e Pi (venv, requirements, sul Pi anche il
container Axelera). È idempotente: se l'ambiente c'è già non fa niente. Lo stadio non conosce il
dry-run, quindi con `DRY_RUN = True` la cella non lancia niente.

In [ ]:
if DRY_RUN:
    print("DRY_RUN attivo: provisioning saltato")
else:
    hydra(stage="provision", hardware=["jetson_orin", "rpi5"])

## 10. Export sulla board (TensorRT, Axelera)

Compila gli artefatti che non si spostano direttamente sulla board di
destinazione: engine TensorRT per workstation e Jetson, modelli Axelera sul Pi.

In [ ]:
if "tensorrt" in WKS_BACKENDS:
    hydra(stage="export", model=MODELS, quantization=QUANTS,
          backend="tensorrt", hardware="wks4_rtx6000")
if "tensorrt" in JETSON_BACKENDS:
    hydra(stage="export", model=MODELS, quantization=QUANTS,
          backend="tensorrt", hardware="jetson_orin")
if "axelera" in RPI_BACKENDS:
    # la quantizzazione la decide il Voyager SDK: basta un solo valore
    hydra(stage="export", model=MODELS, quantization="int8",
          backend="axelera", hardware="rpi5")

## 11. Benchmark — workstation

Misura tutte le combinazioni sulla workstation, in sequenza.

In [ ]:
hydra(stage="benchmark", model=MODELS, quantization=QUANTS,
      backend=WKS_BACKENDS, hardware="wks4_rtx6000",
      compute_target=WKS_COMPUTE, retry_failed = False) # WKS_BACKENDS. MODELS

## 12. Benchmark — Jetson Orin

Un multirun per profilo di potenza, così lo sweep è raggruppato per profilo
e il cambio di `nvpmodel` avviene una volta sola per gruppo.

In [ ]:
for freq in JETSON_FREQS:
    print(f"\n===== jetson_orin @ {freq} =====")
    hydra(stage="benchmark", model=MODELS, quantization=QUANTS,
          backend=JETSON_BACKENDS, hardware="jetson_orin",
          freq_target=freq, compute_target=JETSON_COMPUTE)

## 13. Benchmark — Raspberry Pi 5

Stessa logica del Jetson, un multirun per frequenza della CPU.

In [ ]:
for freq in RPI_FREQS:
    print(f"\n===== rpi5 @ {freq} =====")
    hydra(stage="benchmark", model=MODELS, quantization=QUANTS,
          backend=RPI_BACKENDS, hardware="rpi5",
          freq_target=freq, compute_target=RPI_COMPUTE)

## 14. Riprovare le celle fallite

Le celle `ok` e `skipped` vengono saltate comunque; con `retry_failed`
si rifanno solo quelle fallite. Qui l'esempio è sulla workstation: cambia
`hardware` e gli assi per le altre board.

In [ ]:
hydra(stage="benchmark", model=MODELS, quantization=QUANTS,
      backend=WKS_BACKENDS, hardware="wks4_rtx6000",
      compute_target=WKS_COMPUTE, retry_failed=True)

## 15. Aggregazione

Raccoglie tutti i `results/*.json` in `data.parquet`, il file che leggono
`01_analysis` e `02_figures`.

In [ ]:
subprocess.run([sys.executable, "-m", "tools.aggregate", "--out", "data.parquet"],
               cwd=ROOT, check=True)